# Bangla News Framing: target-aware two-stage training

This notebook builds an interpretable system for comparing how Bangla news articles about the same event frame the government. It predicts sentence–aspect relevance first, predicts government-directed stance only for relevant pairs, aggregates probability-based article profiles, and compares same-event articles.

**Fixed design:** `csebuetnlp/banglabert`; six fixed political-news aspects; labels `NR`, `N`, `GF`, and `GC`; event-grouped splits; and no source, URL, article label, or identifier features in the sentence models. The annotations are AI-assisted and must not be presented as a fully human-verified gold standard.

## 1. Environment setup and reproducibility

Select a GPU runtime in Colab before running this section. The package command is rerunnable: packages already available in the runtime are retained when they satisfy the requirement.

In [ ]:
%pip install -q torch transformers datasets accelerate evaluate scikit-learn pandas numpy matplotlib seaborn pyarrow joblib

In [ ]:
import importlib.metadata as importlib_metadata
import json
import os
import platform
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from transformers import set_seed as set_transformers_seed

PACKAGE_NAMES = [
    "torch", "transformers", "datasets", "accelerate", "evaluate",
    "scikit-learn", "pandas", "numpy", "matplotlib", "seaborn",
    "pyarrow", "joblib",
]
PACKAGE_VERSIONS = {
    name: importlib_metadata.version(name) for name in PACKAGE_NAMES
}

print(f"Python: {platform.python_version()}")
for name, version in PACKAGE_VERSIONS.items():
    print(f"{name}: {version}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA runtime reported by PyTorch: {torch.version.cuda}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is available. In Colab, choose Runtime > Change runtime "
        "type > GPU, reconnect, and rerun the notebook from the beginning."
    )

## 2. Configuration, Drive paths, and deterministic seeds

All paths and initial hyperparameters are defined once here. Colab displays the folder as **My Drive**, while its mounted filesystem path is `/content/drive/MyDrive`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

SEED = 42
PROJECT_ROOT = Path("/content/drive/MyDrive/bangla-news-framing")
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PATH = RAW_DATA_DIR / "banglabias_sentence_annotations_completed.csv"
OUTPUT_DIR = PROJECT_ROOT / "artifacts"

CONFIG = {
    "model_name": "csebuetnlp/banglabert",
    "seed": SEED,
    "data_path": str(DATA_PATH),
    "output_dir": str(OUTPUT_DIR),
    "max_length": 128,
    "learning_rate": 2e-5,
    "train_batch_size": 16,
    "eval_batch_size": 32,
    "max_epochs": 5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.10,
    "early_stopping_patience": 2,
}

def set_all_seeds(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)
    set_transformers_seed(seed)

set_all_seeds(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Place the completed CSV in "
        "My Drive/bangla-news-framing/data/raw and rerun this cell."
    )

ENVIRONMENT = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0),
    "pytorch_cuda": torch.version.cuda,
    "packages": PACKAGE_VERSIONS,
}

with (OUTPUT_DIR / "config.json").open("w", encoding="utf-8") as file:
    json.dump(CONFIG, file, ensure_ascii=False, indent=2)
with (OUTPUT_DIR / "environment.json").open("w", encoding="utf-8") as file:
    json.dump(ENVIRONMENT, file, ensure_ascii=False, indent=2)

print(json.dumps(CONFIG, indent=2, ensure_ascii=False))
print(f"Artifacts will be saved to: {OUTPUT_DIR}")

## 3. Load and strictly validate the annotation CSV

The source CSV remains unchanged. Empty strings are preserved during loading, and assertions stop execution if the file differs from the authoritative schema or expected dataset statistics.

In [ ]:
EXPECTED_COLUMNS = [
    "article_id",
    "event_name",
    "news_headline",
    "news_source",
    "publication_date",
    "source_link",
    "article_stance",
    "sentence_id",
    "sentence_index",
    "sentence_section",
    "sentence_text",
    "a1_policy_administration",
    "a2_accountability_justice",
    "a3_security_civil_rights",
    "a4_democracy_mobilization",
    "a5_economy_development",
    "a6_welfare_social_environment",
    "stance_holder",
    "is_quote",
]
ASPECT_COLUMNS = EXPECTED_COLUMNS[11:17]
ALLOWED_ASPECT_LABELS = {"NR", "N", "GF", "GC"}
ALLOWED_ARTICLE_LABELS = {"Govt critique", "Neutral", "Govt leaning"}
ALLOWED_SECTIONS = {"headline", "lead", "body"}
ALLOWED_STANCE_HOLDERS = {
    "journalist", "government", "opposition", "law_enforcement",
    "court", "expert", "activist", "citizen", "victim_or_family",
    "organization", "unclear",
}

df = pd.read_csv(DATA_PATH, encoding="utf-8", keep_default_na=False)
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns from {DATA_PATH}")
display(pd.DataFrame({"dtype": df.dtypes.astype(str)}))

In [ ]:
def blank_count(frame: pd.DataFrame) -> int:
    return int(frame.astype(str).apply(lambda column: column.str.strip().eq("").sum()).sum())

assert df.columns.tolist() == EXPECTED_COLUMNS, (
    "Unexpected schema or column order.\n"
    f"Expected: {EXPECTED_COLUMNS}\nFound: {df.columns.tolist()}"
)
assert df.shape == (5308, 19), f"Expected shape (5308, 19), found {df.shape}"
assert df["article_id"].nunique() == 118, "Expected 118 unique articles"
assert df["event_name"].nunique() == 26, "Expected 26 unique events"
assert df["sentence_id"].duplicated().sum() == 0, "Duplicate sentence_id values found"
assert df["sentence_text"].str.strip().ne("").all(), "Blank sentence_text found"
assert blank_count(df) == 0, "Unexpected blank cells found in the completed dataset"
assert set(df["article_stance"].unique()) == ALLOWED_ARTICLE_LABELS
assert set(df["sentence_section"].unique()) == ALLOWED_SECTIONS
assert set(df["stance_holder"].unique()) == ALLOWED_STANCE_HOLDERS
assert set(df["is_quote"].unique()) == {0, 1}
assert df.groupby("article_id")["event_name"].nunique().max() == 1
assert df.groupby("article_id")["article_stance"].nunique().max() == 1

for column in ASPECT_COLUMNS:
    found_labels = set(df[column].unique())
    assert found_labels == ALLOWED_ASPECT_LABELS, (
        f"{column}: expected {sorted(ALLOWED_ASPECT_LABELS)}, "
        f"found {sorted(found_labels)}"
    )

article_six_dates = set(
    df.loc[df["article_id"].astype(str).eq("6"), "publication_date"].astype(str)
)
assert article_six_dates == {"2023-09-23"}, (
    f"Article 6 must have corrected date 2023-09-23; found {article_six_dates}"
)

article_rows = df.drop_duplicates("article_id")
audit_summary = {
    "rows": len(df),
    "columns": df.shape[1],
    "articles": int(df["article_id"].nunique()),
    "events": int(df["event_name"].nunique()),
    "duplicate_sentence_ids": int(df["sentence_id"].duplicated().sum()),
    "blank_sentence_texts": int(df["sentence_text"].str.strip().eq("").sum()),
    "blank_cells": blank_count(df),
}
display(pd.Series(audit_summary, name="value").to_frame())
print("All schema and integrity assertions passed.")

## 4. Initial data audit and class imbalance

Article labels are counted once per article. Aspect labels are counted over sentence rows. Accuracy will not be treated as the primary metric because `NR` dominates relevance and `N` dominates relevant-pair stance.

In [ ]:
article_label_counts = (
    article_rows["article_stance"]
    .value_counts()
    .reindex(["Govt leaning", "Neutral", "Govt critique"])
)
section_counts = (
    df["sentence_section"]
    .value_counts()
    .reindex(["headline", "lead", "body"])
)
aspect_label_counts = pd.DataFrame({
    column: df[column].value_counts() for column in ASPECT_COLUMNS
}).T.reindex(columns=["NR", "N", "GF", "GC"]).fillna(0).astype(int)
aspect_label_percentages = aspect_label_counts.div(
    aspect_label_counts.sum(axis=1), axis=0
).mul(100).round(2)

assert article_label_counts.to_dict() == {
    "Govt leaning": 33, "Neutral": 45, "Govt critique": 40
}
assert section_counts.to_dict() == {"headline": 118, "lead": 118, "body": 5072}
assert aspect_label_counts.sum().to_dict() == {
    "NR": 24630, "N": 6882, "GF": 108, "GC": 228
}

print("Article labels (one row per article)")
display(article_label_counts.rename("count").to_frame())
print("Sentence sections")
display(section_counts.rename("count").to_frame())
print("Aspect-label counts")
display(aspect_label_counts)
print("Aspect-label percentages")
display(aspect_label_percentages)

In [ ]:
sentence_lengths = df["sentence_text"].str.len()
length_summary = sentence_lengths.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).round(2)
print("Sentence length in Unicode characters")
display(length_summary.rename("characters").to_frame())

for section in ["headline", "lead", "body"]:
    print(f"Examples — {section}")
    display(
        df.loc[df["sentence_section"].eq(section), ["sentence_id", "sentence_text"]]
        .head(3)
        .style.set_properties(subset=["sentence_text"], **{"text-align": "left"})
    )

In [ ]:
plot_data = (
    aspect_label_counts
    .rename_axis("aspect_column")
    .reset_index()
    .melt(id_vars="aspect_column", var_name="label", value_name="count")
)
plt.figure(figsize=(13, 5))
sns.barplot(
    data=plot_data, x="aspect_column", y="count", hue="label",
    hue_order=["NR", "N", "GF", "GC"],
)
plt.title("Sentence-level labels by aspect")
plt.xlabel("Aspect")
plt.ylabel("Sentence count")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

audit_report = {
    **audit_summary,
    "article_label_counts": {key: int(value) for key, value in article_label_counts.items()},
    "sentence_section_counts": {key: int(value) for key, value in section_counts.items()},
    "aspect_label_counts": {
        aspect: {label: int(value) for label, value in counts.items()}
        for aspect, counts in aspect_label_counts.to_dict(orient="index").items()
    },
    "sentence_length_characters": {
        key: float(value) for key, value in length_summary.items()
    },
}
with (OUTPUT_DIR / "data_audit.json").open("w", encoding="utf-8") as file:
    json.dump(audit_report, file, ensure_ascii=False, indent=2)

print(f"Saved configuration, environment, and audit metadata to {OUTPUT_DIR}")

### Step 1 checkpoint

The runtime, paths, dataset schema, integrity, and documented distributions have been verified.

## 5. Explicit Bangla text preprocessing

Preprocessing is deliberately visible and conservative. The original `sentence_text` is never overwritten; the model receives a new `sentence_text_clean` column.

| Operation | Why it is applied |
|---|---|
| Decode HTML entities and remove actual HTML tags | Remove web-page artifacts without deleting ordinary angle-free text |
| Unicode NFC normalization | Give canonically equivalent Unicode sequences one representation |
| Remove zero-width/BOM artifacts | Prevent invisible characters from creating inconsistent tokens |
| Remove URLs | URLs are not useful stance evidence here; removal is consistent and recorded |
| Collapse line breaks, tabs, and repeated whitespace | Remove formatting differences while preserving word order |

We **do not** remove Bangla stopwords, stem/lemmatize, lowercase, translate, remove all punctuation, or re-segment sentences. Negation, names, numbers, punctuation, sentence IDs, and raw text are preserved. The optional external Bangla normalizer is not used, so every transformation is shown below.

In [ ]:
import html
import re
import unicodedata

HTML_TAG_PATTERN = re.compile(r"<[^>]+>")
URL_PATTERN = re.compile(r"(?:https?://|www\.)\S+", flags=re.IGNORECASE)
WHITESPACE_PATTERN = re.compile(r"\s+")
ZERO_WIDTH_CHARACTERS = "\u200b\u200c\u200d\ufeff"
ZERO_WIDTH_TRANSLATION = str.maketrans("", "", ZERO_WIDTH_CHARACTERS)

def clean_bangla_text(text: str) -> str:
    """Conservatively clean web/Unicode artifacts for BanglaBERT input."""
    cleaned = str(text)
    cleaned = html.unescape(cleaned)
    cleaned = HTML_TAG_PATTERN.sub(" ", cleaned)
    cleaned = unicodedata.normalize("NFC", cleaned)
    cleaned = cleaned.translate(ZERO_WIDTH_TRANSLATION)
    cleaned = URL_PATTERN.sub(" ", cleaned)  # Recorded choice: remove URLs.
    cleaned = WHITESPACE_PATTERN.sub(" ", cleaned).strip()
    return cleaned

raw_sentence_text = df["sentence_text"].copy(deep=True)
df["sentence_text_clean"] = df["sentence_text"].map(clean_bangla_text)
changed_mask = df["sentence_text"].ne(df["sentence_text_clean"])

cleaning_audit = {
    "url_policy": "remove",
    "rows_before": int(len(df)),
    "rows_after": int(len(df)),
    "rows_changed": int(changed_mask.sum()),
    "rows_unchanged": int((~changed_mask).sum()),
    "empty_after_cleaning": int(df["sentence_text_clean"].str.strip().eq("").sum()),
    "raw_rows_containing_url": int(df["sentence_text"].str.contains(URL_PATTERN).sum()),
    "raw_rows_containing_html_tag": int(df["sentence_text"].str.contains(HTML_TAG_PATTERN).sum()),
    "raw_rows_containing_zero_width": int(
        df["sentence_text"].str.contains(f"[{ZERO_WIDTH_CHARACTERS}]", regex=True).sum()
    ),
}

assert len(df) == 5308, "Cleaning must not add or remove sentence rows"
assert df["sentence_text"].equals(raw_sentence_text), "Raw sentence text was modified"
assert df["sentence_text_clean"].str.strip().ne("").all(), (
    "Cleaning produced an empty sentence"
)
assert df["sentence_id"].nunique() == 5308, "Sentence IDs changed during cleaning"

display(pd.Series(cleaning_audit, name="value").to_frame())
examples_to_show = df.loc[
    changed_mask, ["sentence_id", "sentence_text", "sentence_text_clean"]
].head(10)
if examples_to_show.empty:
    print("No rows required cleaning; showing unchanged examples for verification.")
    examples_to_show = df[["sentence_id", "sentence_text", "sentence_text_clean"]].head(5)
display(examples_to_show)

with (OUTPUT_DIR / "preprocessing_audit.json").open("w", encoding="utf-8") as file:
    json.dump(cleaning_audit, file, ensure_ascii=False, indent=2)

## 6. Fixed aspect and label mappings

The model is target-aware: every sentence is paired with one complete Bangla aspect description. Supplying `A1` alone would carry no semantic meaning for BanglaBERT, whereas the full description tells the encoder which government-related target to assess.

In [ ]:
ASPECTS = [
    {
        "aspect_id": "A1",
        "aspect_column": "a1_policy_administration",
        "aspect_name": "Policy and Administration",
        "aspect_text": "সরকারের নীতি, প্রশাসন, সিদ্ধান্ত ও বাস্তবায়ন",
    },
    {
        "aspect_id": "A2",
        "aspect_column": "a2_accountability_justice",
        "aspect_name": "Accountability and Justice",
        "aspect_text": "সরকারের জবাবদিহি, দুর্নীতি নিয়ন্ত্রণ ও ন্যায়বিচার",
    },
    {
        "aspect_id": "A3",
        "aspect_column": "a3_security_civil_rights",
        "aspect_name": "Security and Civil Rights",
        "aspect_text": "সরকারের নিরাপত্তা ব্যবস্থা, আইনশৃঙ্খলা ও নাগরিক অধিকার",
    },
    {
        "aspect_id": "A4",
        "aspect_column": "a4_democracy_mobilization",
        "aspect_name": "Democracy and Mobilization",
        "aspect_text": "সরকারের ভূমিকা সম্পর্কিত নির্বাচন, গণতন্ত্র, রাজনৈতিক অংশগ্রহণ ও আন্দোলন",
    },
    {
        "aspect_id": "A5",
        "aspect_column": "a5_economy_development",
        "aspect_name": "Economy and Development",
        "aspect_text": "সরকারের অর্থনীতি, কর্মসংস্থান, অবকাঠামো ও উন্নয়ন কার্যক্রম",
    },
    {
        "aspect_id": "A6",
        "aspect_column": "a6_welfare_social_environment",
        "aspect_name": "Welfare, Social Issues and Environment",
        "aspect_text": "সরকারের জনকল্যাণ, স্বাস্থ্য, শিক্ষা, সামাজিক সমস্যা ও পরিবেশ ব্যবস্থাপনা",
    },
]
RELEVANCE_LABEL2ID = {"NR": 0, "RELEVANT": 1}
STANCE_LABEL2ID = {"N": 0, "GF": 1, "GC": 2}
ARTICLE_LABEL2ID = {"Govt critique": 0, "Neutral": 1, "Govt leaning": 2}

assert [item["aspect_column"] for item in ASPECTS] == ASPECT_COLUMNS
assert len({item["aspect_id"] for item in ASPECTS}) == 6
assert all(item["aspect_text"].strip() for item in ASPECTS)
display(pd.DataFrame(ASPECTS))

with (OUTPUT_DIR / "aspect_mapping.json").open("w", encoding="utf-8") as file:
    json.dump(ASPECTS, file, ensure_ascii=False, indent=2)
with (OUTPUT_DIR / "label_mappings.json").open("w", encoding="utf-8") as file:
    json.dump(
        {
            "relevance_label2id": RELEVANCE_LABEL2ID,
            "stance_label2id": STANCE_LABEL2ID,
            "article_label2id": ARTICLE_LABEL2ID,
        },
        file, ensure_ascii=False, indent=2,
    )

## 7. Wide annotations to temporary sentence–aspect pairs

The master CSV remains in its permanent wide format. For pooled target-aware training, each of the 5,308 sentences becomes six temporary rows—one per aspect—giving 31,848 sentence–aspect pairs. `NR` maps to relevance 0; `N`, `GF`, and `GC` map to relevance 1. A stance target exists only for gold-relevant pairs.

In [ ]:
PAIR_COLUMNS = [
    "article_id", "event_name", "article_stance", "sentence_id",
    "sentence_index", "sentence_section", "sentence_text_raw",
    "sentence_text_clean", "aspect_id", "aspect_column",
    "aspect_text", "aspect_label", "relevance_label",
    "stance_label", "stance_holder", "is_quote",
]
PAIR_ID_COLUMNS = [
    "article_id", "event_name", "article_stance", "sentence_id",
    "sentence_index", "sentence_section", "sentence_text",
    "sentence_text_clean", "stance_holder", "is_quote", "_sentence_order",
]

aspect_lookup = pd.DataFrame(ASPECTS)
aspect_lookup["_aspect_order"] = np.arange(len(aspect_lookup))
wide_for_pairs = df.assign(_sentence_order=np.arange(len(df)))

long_df = wide_for_pairs.melt(
    id_vars=PAIR_ID_COLUMNS,
    value_vars=ASPECT_COLUMNS,
    var_name="aspect_column",
    value_name="aspect_label",
).merge(
    aspect_lookup[["aspect_id", "aspect_column", "aspect_text", "_aspect_order"]],
    on="aspect_column",
    how="left",
    validate="many_to_one",
)
long_df = (
    long_df
    .sort_values(["_sentence_order", "_aspect_order"], kind="stable")
    .drop(columns=["_sentence_order", "_aspect_order"])
    .rename(columns={"sentence_text": "sentence_text_raw"})
    .reset_index(drop=True)
)
long_df["relevance_label"] = long_df["aspect_label"].ne("NR").astype("int8")
long_df["stance_label"] = long_df["aspect_label"].where(
    long_df["relevance_label"].eq(1), None
)
long_df = long_df[PAIR_COLUMNS]

assert len(long_df) == 5308 * 6 == 31848
assert long_df["sentence_id"].nunique() == 5308
assert long_df.groupby("sentence_id").size().eq(6).all()
assert long_df[["sentence_id", "aspect_id"]].duplicated().sum() == 0
assert long_df["relevance_label"].value_counts().to_dict() == {0: 24630, 1: 7218}
assert long_df["stance_label"].value_counts().to_dict() == {"N": 6882, "GC": 228, "GF": 108}
assert long_df.loc[long_df["aspect_label"].eq("NR"), "stance_label"].isna().all()
assert long_df.loc[long_df["aspect_label"].ne("NR"), "stance_label"].notna().all()

print(f"Rows before transformation: {len(df):,}")
print(f"Rows after transformation:  {len(long_df):,}")
print("Relevance labels:")
display(long_df["relevance_label"].value_counts().sort_index().rename("count").to_frame())
print("Gold-relevant stance labels:")
display(long_df["stance_label"].value_counts().rename("count").to_frame())
display(long_df.head(12))

## 8. Deterministic event-grouped train/validation/test split

A sentence-level random split would leak event-specific people, facts, and phrasing. Instead, the 26 sorted event names are shuffled once with seed 42 and assigned as 18 train, 4 validation, and 4 untouched test events (approximately 69.2%/15.4%/15.4% by event). This is a deterministic grouped development split—not exact stratification—and its observed label distributions are reported. No decision later may be selected using test performance.

In [ ]:
SPLIT_ORDER = ["train", "validation", "test"]
event_names = sorted(long_df["event_name"].unique().tolist())
split_rng = random.Random(SEED)
split_rng.shuffle(event_names)

n_test_events = round(0.15 * len(event_names))
n_validation_events = round(0.15 * len(event_names))
test_events = set(event_names[:n_test_events])
validation_events = set(event_names[n_test_events:n_test_events + n_validation_events])
train_events = set(event_names[n_test_events + n_validation_events:])
EVENTS_BY_SPLIT = {
    "train": train_events,
    "validation": validation_events,
    "test": test_events,
}
event_to_split = {
    event: split_name
    for split_name, split_events in EVENTS_BY_SPLIT.items()
    for event in split_events
}
long_df["split"] = long_df["event_name"].map(event_to_split)
df["split"] = df["event_name"].map(event_to_split)

assert len(train_events) == 18 and len(validation_events) == 4 and len(test_events) == 4
assert train_events.isdisjoint(validation_events)
assert train_events.isdisjoint(test_events)
assert validation_events.isdisjoint(test_events)
assert set(event_to_split) == set(event_names)
assert long_df["split"].notna().all() and df["split"].notna().all()

article_split_table = df.drop_duplicates("article_id")[["article_id", "split", "article_stance"]]
article_sets = {
    split_name: set(article_split_table.loc[article_split_table["split"].eq(split_name), "article_id"])
    for split_name in SPLIT_ORDER
}
assert article_sets["train"].isdisjoint(article_sets["validation"])
assert article_sets["train"].isdisjoint(article_sets["test"])
assert article_sets["validation"].isdisjoint(article_sets["test"])

for split_name in SPLIT_ORDER:
    split_pairs = long_df.loc[long_df["split"].eq(split_name)]
    split_articles = article_split_table.loc[article_split_table["split"].eq(split_name)]
    assert set(split_pairs["aspect_label"].unique()) == ALLOWED_ASPECT_LABELS, (
        f"Not all pair labels are present in {split_name}"
    )
    assert set(split_articles["article_stance"].unique()) == ALLOWED_ARTICLE_LABELS, (
        f"Not all article labels are present in {split_name}"
    )

split_overview = pd.DataFrame({
    "events": {name: len(events) for name, events in EVENTS_BY_SPLIT.items()},
    "articles": article_split_table["split"].value_counts(),
    "sentences": df["split"].value_counts(),
    "sentence_aspect_pairs": long_df["split"].value_counts(),
}).reindex(SPLIT_ORDER)
article_distribution = pd.crosstab(
    article_split_table["split"], article_split_table["article_stance"]
).reindex(SPLIT_ORDER)
pair_distribution = pd.crosstab(
    long_df["split"], long_df["aspect_label"]
).reindex(index=SPLIT_ORDER, columns=["NR", "N", "GF", "GC"])

display(split_overview)
print("Article-label distribution")
display(article_distribution)
print("Sentence–aspect label distribution")
display(pair_distribution)
print("Zero event overlap and zero article overlap: PASSED")

In [ ]:
split_assignments = {
    "seed": SEED,
    "strategy": "sorted event names shuffled once; 18/4/4 grouped allocation",
    "test_set_policy": "untouched until all model and threshold choices are frozen",
    "splits": {
        split_name: {
            "event_names": sorted(EVENTS_BY_SPLIT[split_name]),
            "article_ids": sorted(str(value) for value in article_sets[split_name]),
        }
        for split_name in SPLIT_ORDER
    },
}
with (OUTPUT_DIR / "split_assignments.json").open("w", encoding="utf-8") as file:
    json.dump(split_assignments, file, ensure_ascii=False, indent=2)

print("Split assignments saved. The processed pair file is saved after token-length inspection below.")

## 9. BanglaBERT tokenizer and embedding dimensions

The tokenizer constructs the paired input and special tokens automatically from `(sentence_text_clean, aspect_text)`. With dynamic padding, a batch is padded only to its longest member.

The key dimensions are:

- `input_ids`: `[batch_size, sequence_length]` integer token IDs.
- token lookup embeddings: `[batch_size, sequence_length, embedding_size]`; one initial vector per token.
- contextual embeddings (`last_hidden_state`): `[batch_size, sequence_length, hidden_size]`; every token vector has attended to the paired sentence/aspect context. Some architectures project from `embedding_size` to `hidden_size`, so both values are printed.
- pooled `[CLS]` representation: `[batch_size, hidden_size]`; the later classification head maps each row to 2 relevance logits or 3 stance logits.

The code prints actual values from the downloaded model configuration rather than assuming a hidden size.

In [ ]:
from transformers import AutoConfig, AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
banglabert_config = AutoConfig.from_pretrained(CONFIG["model_name"])

print(f"Tokenizer class: {tokenizer.__class__.__name__}")
print(f"Vocabulary size: {banglabert_config.vocab_size:,}")
embedding_size = getattr(banglabert_config, "embedding_size", banglabert_config.hidden_size)
print(f"Token embedding size: {embedding_size}")
print(f"Transformer hidden size: {banglabert_config.hidden_size}")
print(f"Transformer layers: {banglabert_config.num_hidden_layers}")
print(f"Attention heads: {banglabert_config.num_attention_heads}")
print(f"Maximum model positions: {banglabert_config.max_position_embeddings}")

inspection_rows = long_df.iloc[[0, 7]][["sentence_text_clean", "aspect_text"]]
inspection_batch = tokenizer(
    inspection_rows["sentence_text_clean"].tolist(),
    inspection_rows["aspect_text"].tolist(),
    truncation=True,
    max_length=CONFIG["max_length"],
    padding=True,  # For inspection only; training will use dynamic padding in its collator.
    return_tensors="pt",
)
print("Encoded tensor shapes:")
for key, value in inspection_batch.items():
    print(f"  {key}: {tuple(value.shape)}")
print("First paired input with model special tokens:")
print(tokenizer.convert_ids_to_tokens(inspection_batch["input_ids"][0]))

In [ ]:
def pair_token_lengths(frame: pd.DataFrame, batch_size: int = 512) -> list[int]:
    """Measure untruncated paired-input lengths without retaining token IDs."""
    lengths = []
    for start in range(0, len(frame), batch_size):
        batch = frame.iloc[start:start + batch_size]
        encoded = tokenizer(
            batch["sentence_text_clean"].tolist(),
            batch["aspect_text"].tolist(),
            add_special_tokens=True,
            truncation=False,
            padding=False,
            return_length=True,
        )
        lengths.extend(encoded["length"])
    return [int(length) for length in lengths]

long_df["token_length"] = pair_token_lengths(long_df)
truncated_mask = long_df["token_length"].gt(CONFIG["max_length"])
token_length_summary = long_df["token_length"].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99]
).round(2)
display(token_length_summary.rename("paired_tokens").to_frame())
print(
    f"Pairs longer than max_length={CONFIG['max_length']}: "
    f"{int(truncated_mask.sum()):,}/{len(long_df):,} "
    f"({truncated_mask.mean():.2%})"
)

long_df.to_parquet(OUTPUT_DIR / "processed_sentence_aspect_pairs.parquet", index=False)
print(f"Saved processed pairs to {OUTPUT_DIR / 'processed_sentence_aspect_pairs.parquet'}")

In [ ]:
base_model = AutoModel.from_pretrained(CONFIG["model_name"]).to("cuda")
base_model.eval()
gpu_inspection_batch = {key: value.to("cuda") for key, value in inspection_batch.items()}

with torch.inference_mode():
    # Word/token lookup vectors before transformer contextualization.
    token_lookup_embeddings = base_model.get_input_embeddings()(
        gpu_inspection_batch["input_ids"]
    )
    # Contextual vectors after all transformer layers.
    contextual_embeddings = base_model(**gpu_inspection_batch).last_hidden_state
    cls_embeddings = contextual_embeddings[:, 0, :]

print(f"Token IDs [batch, tokens]: {tuple(gpu_inspection_batch['input_ids'].shape)}")
print(
    "Lookup embeddings [batch, tokens, hidden]: "
    f"{tuple(token_lookup_embeddings.shape)}"
)
print(
    "Contextual embeddings [batch, tokens, hidden]: "
    f"{tuple(contextual_embeddings.shape)}"
)
print(f"[CLS] embeddings [batch, hidden]: {tuple(cls_embeddings.shape)}")
assert token_lookup_embeddings.shape[-1] == embedding_size
assert contextual_embeddings.shape[-1] == banglabert_config.hidden_size
assert cls_embeddings.shape == (len(inspection_rows), banglabert_config.hidden_size)

# Free this inspection-only model before loading the relevance classifier next.
del base_model, gpu_inspection_batch, token_lookup_embeddings, contextual_embeddings, cls_embeddings
torch.cuda.empty_cache()

### Step 2 checkpoint

The notebook now demonstrates preprocessing explicitly, creates and validates 31,848 target-aware pairs, freezes a leakage-safe event split, audits token truncation, and explains the model's embedding dimensions using real tensors. The next increment trains and evaluates the weighted Stage 1 relevance classifier.